In [1]:
'''objetivos:
comparar resposta ao ruído para FF de 0.1 a 40% usando os métodos:
dixon
multieco 3 pontos
multi-interferência 7 pontos
ann

materiais e métodos
datasets de teste:
FF: 1% a 40%, step de 1% (40 steps), randomizado uniforme, 10 iterações por step
T2* 0.9, 1.3, 2.1 ppm randomizados na gaussiana limitados entre 0 e 1
T2* 4.2 e 5.3 ppm fixos
T2* 4.7 (agua) ppm randomizado na gaussiana limitado entre 0 e 1
ruído randomizado adicionado em relação ao primeiro ponto:
teste com SNR = 200
teste com SNR = 100
teste com SNR = 50
teste com SNR = 25

dataset de treino do ann:
dataset padrão
FF: 1% a 100%, step de 0.1% (1000 steps), linear, 50 iterações por step
T2* 0.9, 1.3, 2.1 ppm randomizados na gaussiana limitados entre 0 e 1
T2* 4.2 e 5.3 ppm fixos
T2* 4.7 (agua) ppm randomizado na gaussiana limitado entre 0 e 1
ruído randomizado uniforme adicionado em relação ao primeiro ponto:
SNR [200 25]

MÉTRICAS
standard deviation of the pairwise difference and correlation coefficient between real and estimation
sensitivity, specificity 
	(threshold 6.25%? 
	two-sided confidence intervals 95%?
	bias-corrected and accelerated intervals from bootstrap distribution of 1000 resampled datasets?
	thresholds 5,6,7,8%?)
regression slopes and intercepts
	null hypostesis: slope 1, intercept 0
	two-tailed t test, alpha=0.05
'''

'objetivos:\ncomparar resposta ao ruído para FF de 0.1 a 40% usando os métodos:\ndixon\nmultieco 3 pontos\nmulti-interferência 7 pontos\nann\n\nmateriais e métodos\ndatasets de teste:\nFF: 1% a 40%, step de 1% (40 steps), randomizado uniforme, 10 iterações por step\nT2* 0.9, 1.3, 2.1 ppm randomizados na gaussiana limitados entre 0 e 1\nT2* 4.2 e 5.3 ppm fixos\nT2* 4.7 (agua) ppm randomizado na gaussiana limitado entre 0 e 1\nruído randomizado adicionado em relação ao primeiro ponto:\nteste com SNR = 200\nteste com SNR = 100\nteste com SNR = 50\nteste com SNR = 25\n\ndataset de treino do ann:\ndataset padrão\nFF: 1% a 100%, step de 0.1% (1000 steps), linear, 50 iterações por step\nT2* 0.9, 1.3, 2.1 ppm randomizados na gaussiana limitados entre 0 e 1\nT2* 4.2 e 5.3 ppm fixos\nT2* 4.7 (agua) ppm randomizado na gaussiana limitado entre 0 e 1\nruído randomizado uniforme adicionado em relação ao primeiro ponto:\nSNR [200 25]\n\nMÉTRICAS\nstandard deviation of the pairwise difference and corr

In [2]:
# INICIALIZAÇÃO DAS BIBLIOTECAS
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from scipy.stats import truncnorm
import arff
import itertools


####### FUNÇÕES PARA OBTENÇÃO DO SINAL ######

def get_truncated_normal(mean=0, sd=1, low=0, upp=10):
    return truncnorm(
        (low - mean) / sd, (upp - mean) / sd, loc=mean, scale=sd)

def set_random_rho(ff):
    '''
    Returns the Proton Density values rho_fat, rho_water for a given Fat fraction ff
    parameters: ff     -> float 0>1 fat fraction
    returns:    rho_fat, rho_water
    '''
    rho_fat = np.random.rand()
    rho_water = (rho_fat/ff)-rho_fat
    return(rho_fat, rho_water)

def discrete_timeseries(n_echoes):
    '''
    This function calculates the echo times for n_echoes in the GRE IP-OP sequence, considering TE = 0.0011515, 
    which corresponds to water (4.7 ppm) and fat (1.3 ppm) interference at 3 Tesla.
    parameters: n_echoes     -> integer >0, number of equally spaced echoes
    returns:    echo_times
     '''
    
    # TE acquisition times for n echoes
    echo_times = np.linspace(0.0011515, 0.0011515*n_echoes, n_echoes, endpoint=True)
    return(echo_times)

def gen_GRE_noise(echo_times, rho_fat, rho_water, t2fat09=0.083, t2fat13=0.061, t2fat21=0.052, t2fat275=0.051, 
                  t2fat42=0.0254, t2fat53=0.0387, t2water47=0.027, noise_level=0.00):
    '''
    This function calculates the GRE IP-OP signal from human liver, for any number of TEs,
    considering a low flip angle (Taylor approx. around zero) and 3T.
    It is based on studies by Hamilton (T2* and characteristic ppm frequency values).
    parameters: echo times -> array with TEs
                rho_fat    -> float >0, fat proton density
                rho_water  -> float >0, water proton density
                t2star     -> array >0, T2* values for fat peaks at 0.9, 1.3., 2.1, 2.75, 4.2 and 5.3ppm (fat) and 
                                        water at 4.7 ppm. Standard values are [0.030, 0.040, 0.051, 0.052, 0.062, 0.083, 0.023]
                noise_sd   -> float >0, noise level (1/SNR) as proportion of the first signal point intensity 
                                        (default is 0%)
    returns:    Sabs -> float > 0, absolute signal intensity in arbitrary units (equivalent to ydata)
    '''
# rho_fat is a general variable that multiplies intrinsic fat constants (Hamilton)
    rho = np.array([rho_fat*0.088, rho_fat*0.7, rho_fat*0.12, rho_fat*0.006, rho_fat*0.039, rho_fat*0.047, rho_water])
    
    # individual T2* values
    t2star = np.array([t2fat09, t2fat13, t2fat21, t2fat275, t2fat42, t2fat53, t2water47])
    
    # frequency in parts per million for each moiety
    freq_ppm = np.array([0.9, 1.3, 2.1, 2.75, 4.2, 5.3, 4.7])
    
    freq = np.zeros(len(freq_ppm))
    magField = np.array(3.0)      # magnetic field, Tesla (if change, remember to change echo spacing)
    hydroGyro = np.array(42.57e6) # Hydrogen gyromagnetic ratio, Hz/T
    
    for n, fppm in enumerate(freq_ppm):
        freq[n] = fppm*magField*hydroGyro/1e6
    
    # calculate the complex signal for each moiety    
    S = (len(freq_ppm),len(echo_times))
    S = np.zeros(S,dtype='complex')
    
    for n, moieties in enumerate(freq_ppm):
        for t,TE in enumerate(echo_times):
            S[n,t] = rho[n]*np.exp(-TE/t2star[n])*np.exp(2.0*np.pi*1j*freq[n]*TE)         
        
    #obtain sum & add noise
    Ssum= (len(echo_times))
    Ssum=np.zeros(Ssum,dtype='complex')
    Ssum = np.sum(S, axis=0)
    
    for t,TE in enumerate(echo_times):
        Ssum[t] = Ssum[t].real+Ssum[0].real*noise_level*np.random.normal()+1j*(Ssum[t].imag+Ssum[0].imag*noise_level*np.random.normal())
 
    #obtain absolute value
    Sabs= (len(echo_times))
    Sabs=np.zeros(Sabs,dtype='complex')
    Sabs = np.absolute(Ssum)
    return (Sabs)



################### FUNÇÕES PARA FITTING ###########

def fit_multiinterf(echo_times, rho_fat, rho_water, t2star):
    '''
    This function fits the GRE IP-OP signal from human liver, for any number of TEs,
    considering a low flip angle (Taylor approx. around zero) and 3T.
    It is based on studies by Hamilton (T2* and characteristic ppm frequency values).
    parameters: echo times -> array with TEs
                rho_fat    -> float >0, fat proton density
                rho_water  -> float >0, water proton density
    returns:    Sabs -> float > 0, absolute signal intensity in arbitrary units (equivalent to ydata)
    '''
    
    echo_times = np.array(echo_times)
    rho_fat = np.array(rho_fat)
    rho_water = np.array(rho_water)
    t2star = np.array(t2star)
    
    #cn is the fractional proton density of each fat moiety (Yokoo) (see freq_ppm)
    cn = np.array([0.12, 0.7, 0.088,]) 
    
    #frequency in parts per million for each fat moiety
    ppm_fat = np.array([2.1, 1.3, 0.9])
    ppm_water = np.array(4.7)
    
    magField = np.array(3.0) #magnetic field, Tesla (if change, remember to change echo spacing)
    hydroGyro = np.array(42.577e6) #Hydrogen gyromagnetic ratio, Hz/T
    freq_water = np.array(ppm_water*magField*hydroGyro/1e6) #water frequency in Hertz
    
    #relative frequency for each fat to that of water
    freqr_fat = np.zeros(len(ppm_fat))
    for n, fppm in enumerate(ppm_fat):
        freqr_fat[n] = freq_water-(fppm*magField*hydroGyro/1e6)
    
    #calculate the signal for each moiety    
    S = (len(freqr_fat),len(echo_times))
    S = np.zeros(S,dtype='complex')
        #three fat signals
    for n, moieties in enumerate(freqr_fat):
        for t,TE in enumerate(echo_times):
            S[n,t] = cn[n]*np.exp(2.0*np.pi*1j*freqr_fat[n]*TE)
    
    Ssum= len(echo_times)
    Ssum = np.zeros(Ssum)
    Ssum = np.sum(S, axis=0) #sum of fat signals
    Sabs = np.absolute(rho_water + rho_fat*Ssum) #multiply rho_fat and add rho_water, take absolute value
    Sfinal = len(echo_times)
    Sfinal = np.zeros(Sfinal)
    for t,TE in enumerate(echo_times):
        Sfinal[t] = Sabs[t]*np.exp(-TE/t2star) #exponential decay
    
    return (Sfinal)

def calc_fatfrac(fitted_parameters):
    '''
    Returns the Fat Fraction ff  for values rho_fat (fitted_parameters[0]) and rho_water (fitted_parameters[1])
    parameters: fitted parameters -> list or tuple, where rho_fat = fitted_parameters[0] and rho_water fitted_parameters[1])
    returns:    ff - > fat fraction
    '''
    ff =  fitted_parameters[0] / (fitted_parameters[0] + fitted_parameters[1])
    return(ff)

######### FUNÇÕES PARA OBTENÇÃO DOS DESCRITORES #######
def get_pairs(a, n):
    '''Retorna a combinação de n pares consecutivos a um dado ponto em uma sequência.
    Ex:
    a = [1,2,3,4]
    n = 2
    res = [(1,2), (1,3), (2,3), (2,4), (3,4)]
    '''
    nmax = len(a)
    if n >= nmax:
        n = nmax-1
        
    res_esq = []
    res_dir = []

    m = 1
    for i in range(nmax-n):
        res_esq.append(np.repeat(a[i],n))
        res_dir.append(a[m:m+n])
        m= m+1

    k=1
    for i in range(n-1):
        res_esq.append(np.repeat(a[nmax-n+i],n-k))
        res_dir.append(a[nmax-n+k:nmax+i])
        k=k+1

    res_esq = [item for sublist in res_esq for item in sublist] #flatten
    res_dir = [item for sublist in res_dir for item in sublist] #flatten

    res = np.dstack((res_esq,res_dir)) #junta os dados 
    res = res[0] #remove array externo
    res_tuple = [tuple(l) for l in res] 

    return res_tuple


def calc_dist_angle(p0,p1):
    '''
    funciona para pontos dentro do dominio x:[0,1], y:[0,1] (supondo o sinal de modulo normalizado pela norma de frobenius)
    Returns the distance and angle between two GRE IP-OP points normalized between [0,1], considering  the vertical Y axis as zero-reference 
    for the angle.
    parameters: p0 , p1 -> y-value (intensity) of the points between [0,1]
    returns:    dist_norm, theta_rad_norm - > distance and angle between [0,1]
    '''
    #gera 2 pontos de TE e subtrai o primeiro do segundo (obter a base de tempo)
    #o multiplicador de 1000 lineariza a função cosseno (verificar essa explicação!)
    #time = 1000*(discrete_timeseries(2))[1]-(discrete_timeseries(2))[0] 
    time = 1
    
    #como não é um triângulo retângulo, vamos usar a lei dos cossenos para obter o ângulo
    #e a distância sai junto
    #para o ângulo a referência de zero grau é a posição 12 horas, paralela ao eixo y

    #obtem lados do triângulo, valor 2 foi usado como altura total em a e b para não dar erro quando p0=1 
    a = np.sqrt(np.square(2-p1)+np.square(time)) #a corresponde ao lado superior
    b = 2-p0 #b corresponde ao lado paralelo ao eixo y
    c = np.sqrt(np.square(p0-p1)+np.square(time)) #c corresponde ao lado que conecta os pontos p0 e p1

    distance = c
    cosA = (np.square(b)+np.square(c)-np.square(a))/(2*b*c) #lei dos cossenos
    theta_rad = np.arccos(cosA)

    #a função deve retornar a distância e ângulo normalizados entre 0 e 1. Uma função normalizada assim é dada por:
    # F(x)[0,1] = (f(x)-min(f(x)))/(max(f(x)-min(f(x)))
    #precisamos então calcular as valores de de distância e ângulo nos extremos.

    #valor minimo que a distancia pode atingir    
    dist_min = time
    #valor máximo que a distancia pode atingir
    dist_max = np.sqrt(1+np.square(time)) 
    #normalização
    dist_norm = (distance-dist_min)/(dist_max-dist_min)

    #para normalizar o ângulo, calcula-se o menor e maior valor do ângulo
    #o menor ocorre quando (p0,p1) = (0,1)
    p0_min = 0
    p1_min = 1
    a_min = np.sqrt(np.square(2-p1_min)+np.square(time)) #a corresponde ao lado superior
    b_min = 2-p0_min #b corresponde ao lado paralelo ao eixo y
    c_min = np.sqrt(np.square(p0_min-p1_min)+np.square(time)) #c corresponde ao lado que conecta os pontos p0 e p1

    cosA_min = (np.square(b_min)+np.square(c_min)-np.square(a_min))/(2*b_min*c_min) #lei dos cossenos
    theta_min = np.arccos(cosA_min)
    #o maior ocorre quando (p0,p1) = (1,0)
    #o cálculo do maior valor pode ser obtido subtraindo 180 graus do resultado do menor ângulo
    theta_max = np.pi-theta_min
    #normalização
    theta_rad_norm = (theta_rad-theta_min)/(theta_max-theta_min)
       
    return dist_norm, theta_rad_norm

def make_FF_arff(dataset):
    '''
    Coloca o dataset no formato arff
    usa import arff
    '''
    obj = {
   'description': u'distance, angle and fat fraction',
   'relation': 'fat fraction ml',
   'attributes': [
       ('distance 1-2', 'REAL'),
       ('angle 1-2', 'REAL'),
       ('distance 2-3', 'REAL'),
       ('angle 2-3', 'REAL'),
       ('distance 3-4', 'REAL'),
       ('angle 3-4', 'REAL'),
       ('distance 4-5', 'REAL'),
       ('angle 4-5', 'REAL'),
       ('distance 5-6', 'REAL'),
       ('angle 5-6', 'REAL'),
       ('distance 6-7', 'REAL'),
       ('angle 6-7', 'REAL'),
       ('fat fraction', 'REAL'),
   ],
   'data': dataset,
    }
    return arff.dumps(obj)



### Test data

In [6]:
####### SETTINGS - MEXA AQUI #########
ff = np.linspace(0.01, 0.4, num=40) # fat fraction, limites, numero de discretizações
ff_iterations = 10 #numero de iterações do mesmo valor de FF
n_pairs = 6 #numero de pares consecutivos, de 1 a 6. Se for colocado maior que o permitido, ele substitui pelo máximo permitido.

######################CÓDIGO############################################
######## parâmetros do sinal ########
total_size = len(ff)*ff_iterations

t2fat09 = get_truncated_normal(0.083, 0.0163, low=0.001, upp=1).rvs(total_size) #mean, sd, min, max (para não obter valor negativo)
t2fat13 = get_truncated_normal(0.061, 0.0113, low=0.001, upp=1).rvs(total_size)
t2fat21 = get_truncated_normal(0.052, 0.0298, low=0.001, upp=1).rvs(total_size)
t2fat275 = np.repeat(0.051,total_size)
t2fat42 = np.repeat(0.0254,total_size)
t2fat53 = np.repeat(0.0387,total_size)
t2water47 = get_truncated_normal(0.027, 0.0066, low=0.001, upp=1).rvs(total_size)

snr = 200
noise_values = np.repeat(1/snr,total_size)

######## inicialização dos arrays ######

ff_repeat = np.repeat(ff,ff_iterations) #repete ff para preencher a lista

signal_norm_collection = np.zeros([total_size,7]) #sinal normalizado
distance_collection = np.zeros([total_size,6]) #6 distancias
theta_collection = np.zeros([total_size,6]) #6 angulos
xdata = discrete_timeseries(7) #gerar TEs para 3T


######### gera as curvas #######
n=0
for fat_fraction in ff_repeat:
    rho_fat, rho_water = set_random_rho(fat_fraction) #obter rhos aleatorios que atendam a uma determinada FF
    signal = gen_GRE_noise(xdata, rho_fat, rho_water, t2fat09[n], t2fat13[n], t2fat21[n], t2fat275[n], 
                           t2fat42[n], t2fat53[n], noise_values[n]) #obter sinal GRE
    signal_norm = signal/np.linalg.norm(signal) #normalizar os sinais, norma de frobenius
    signal_norm_collection[n,:] = signal_norm #colocar os sinais numa coleção, pontos 0:6
    n = n+1

########### calcula os descritores #########    

ff_collection = []    

for item in signal_norm_collection:

    signal_pairs = get_pairs(item, n_pairs)
    
    for x in signal_pairs:
        ff_collection.append(calc_dist_angle(x[0],x[1]))

ff_collection = np.asarray(ff_collection)
ff_collection = np.reshape(ff_collection,(total_size,2*len(signal_pairs)))    
 
ff_collection = np.c_[ff_collection,ff_repeat]

########### salva csv ##########

import pandas as pd 
df = pd.DataFrame(ff_collection)
df.to_csv("abstract"+str(total_size)+"_"+str(2*len(signal_pairs))+"descrit_ff1a40_snr"+str(snr)+"_teste.csv", header=False, index=False)

######### salva arff ##########
# ff_arff = make_FF_arff(ff_collection)
# ff_arff = arff.dumps(ff_arff)

# fo = open("testarff.arff", "w+")
# fo.write(ff_arff)

# # Close file
# fo.close()

### Dixon

### Multiecho

### Multiinterference